# LLM Portfolio Brief Prototype

Goal: prepare a compact, grounded prompt from current model outputs. Keep the LLM restricted to explaining existing signals rather than inventing trades.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
import json
import pandas as pd

from ai_models.alert_engine import generate_alerts
from ai_models.regime_detection_model import run_regime_detection_model
from ai_models.risk_detector import run_systemic_risk_detector

regime = run_regime_detection_model(prices_path="data/prices_cache.parquet", treasury_path="data/treasury_yields_cache.parquet")
risk = run_systemic_risk_detector(prices_path="data/prices_cache.parquet", treasury_path="data/treasury_yields_cache.parquet")
alerts = generate_alerts(
    drift_df=pd.DataFrame(),
    regime_df=regime,
    risk_df=risk,
    coverage_stats={"treasury_exists": True, "prices_rows": 50000, "expected_min_price_rows": 50000},
)
alerts.head()

In [ ]:
prompt_payload = {
    "latest_regime": regime.tail(3).to_dict(orient="records"),
    "latest_risk": risk.tail(5).to_dict(orient="records"),
    "latest_alerts": alerts.head(5).to_dict(orient="records"),
    "instruction": "Write a concise portfolio brief using only the supplied facts. Do not invent new numbers or securities.",
}
print(json.dumps(prompt_payload, indent=2, default=str)[:4000])

In [ ]:
# Example OpenAI call placeholder.
# Uncomment after adding your preferred SDK and API key.
# from openai import OpenAI
# client = OpenAI()
# response = client.responses.create(
#     model="gpt-5-mini",
#     input=json.dumps(prompt_payload, default=str),
# )
# print(response.output_text)